# HAKE-MER — RoBERTa +M1 and +M1+M2 (multi-backbone control)

**Backbone:** RoBERTa-base · same protocol as DistilBERT ablations (batch 16, LR 5e-5, 4 epochs, seeds 42/123/456).

Completes objective~2: compare hierarchical stack on a second PLM without re-running M3/M4.

DistilBERT references: Step~0 **0.486**, +M1 **0.457**, +M1+M2 **0.505** (F1-macro test).

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_roberta_ablation.sh {TRAIN_FLAGS}

In [ ]:
import json
from pathlib import Path

for name in ("m1_roberta_base_campaign.json", "m1_m2_roberta_base_campaign.json"):
    c = json.loads((Path("reference/artifacts") / name).read_text(encoding="utf-8"))
    m = c["test_aggregate"]["f1_macro"]
    print(name, f"F1-macro test {m['mean']:.4f} ± {m['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files
from pathlib import Path

slug = "roberta_base"
paths = [
    Path(f"reference/artifacts/m1_{slug}_campaign.json"),
    Path(f"reference/artifacts/m1_m2_{slug}_campaign.json"),
]
out_name = "roberta_m1_m1_m2_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in paths:
        zf.write(p, p.name)
    for pat in (f"{slug}_seed*_m1/metrics.json", f"{slug}_seed*_m1_m2/metrics.json"):
        for m in sorted(Path("runs").glob(pat)):
            zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

## Integrity

1. **Download .ipynb** → `reference/training_records/step_roberta_ablation/colab/`
2. Unzip campaign JSON (+ optional metrics) → `reference/artifacts/`
3. Run `python3 scripts/print_roberta_ablation_latex.py` locally and paste rows into ch.~4 table `tab:expe-roberta-ablation`.